# SimCLR — Contrastive Learning of Visual Representations
**Chen et al., ICML 2020**

**Category:** `03-Self-Supervised-Learning / 02-ContrastiveLearning`

SimCLR learns representations by **maximizing agreement between differently augmented views** of the same image via a contrastive loss (NT-Xent) in latent space. No negative mining, no memory bank — just a large batch.

## Key Ideas
| Component | Detail |
|---|---|
| **Augmentation** | RandomCrop + ColorJitter (both required — crop alone lets the network "cheat") |
| **Encoder f(·)** | ResNet-50; outputs representation **h** |
| **Projection head g(·)** | 2-layer MLP → **z** (contrastive loss applied here, not h) |
| **Loss** | NT-Xent: temperature-scaled cross-entropy over 2(N-1) negatives |

> **Why z not h?** The projection head absorbs augmentation-invariance. Using h for downstream tasks gives >10% better accuracy (SimCLR Section 4.2).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Step 1: Augmentation Pipeline
The composition of **RandomCrop + ColorJitter** is strictly required.  
Two independent augmentations of the same image form a **positive pair**.


In [ ]:
class SimCLRAugmentation:
    """Returns two independently augmented views of the same image."""
    def __init__(self, size=32):
        self.transform = T.Compose([
            T.RandomResizedCrop(size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.GaussianBlur(kernel_size=3),
            T.ToTensor(),
            T.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010]),
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)


class CIFAR10SSL(Dataset):
    def __init__(self, transform):
        self.data = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True,
            transform=transform
        )
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx][0]  # drop label

aug = SimCLRAugmentation(size=32)


## Step 2: Encoder + Projection Head
`f(·)` = ResNet-18 (lighter than paper's ResNet-50, suitable for CIFAR-10)  
`g(·)` = 2-layer MLP: dim → 512 → 128


In [ ]:
class SimCLR(nn.Module):
    def __init__(self, base_encoder=torchvision.models.resnet18, proj_dim=128):
        super().__init__()
        # Encoder
        self.encoder = base_encoder(weights=None)
        dim_mlp = self.encoder.fc.in_features          # 512 for ResNet-18
        self.encoder.fc = nn.Identity()                # remove classifier

        # Projection head  g(·): h → z
        self.projector = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, proj_dim),
        )

    def forward(self, x):
        h = self.encoder(x)          # representation
        z = self.projector(h)        # projection
        return h, z


## Step 3: NT-Xent Loss
$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k)/\tau)}$$

- `sim(u,v)` = cosine similarity  
- `τ` = temperature (default 0.5) — lower τ → sharper distribution → harder negatives penalised more


In [ ]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.tau = temperature

    def forward(self, z1, z2):
        N = z1.size(0)
        # L2-normalise
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        # Concatenate: [z1; z2] shape (2N, D)
        z = torch.cat([z1, z2], dim=0)

        # Similarity matrix (2N x 2N)
        sim = torch.mm(z, z.T) / self.tau

        # Mask out self-similarities
        mask = torch.eye(2 * N, device=z.device).bool()
        sim.masked_fill_(mask, -9e15)

        # Positive pair indices: (i, i+N) and (i+N, i)
        labels = torch.arange(N, device=z.device)
        labels = torch.cat([labels + N, labels])   # positives

        loss = F.cross_entropy(sim, labels)
        return loss


## Step 4: Training

In [ ]:
BATCH_SIZE = 256
EPOCHS     = 10
LR         = 3e-4
TEMPERATURE = 0.5

# Dataset with two-view augmentation
class TwoViewDataset(Dataset):
    def __init__(self, aug):
        self.base = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=None)
        self.aug = aug
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, _ = self.base[idx]
        return self.aug(img)

loader = DataLoader(TwoViewDataset(aug), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=2, drop_last=True)

model     = SimCLR().to(device)
criterion = NTXentLoss(TEMPERATURE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

losses = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = []
    for (x1, x2) in loader:
        x1, x2 = x1.to(device), x2.to(device)
        _, z1 = model(x1)
        _, z2 = model(x2)
        loss = criterion(z1, z2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    scheduler.step()
    avg = sum(epoch_loss) / len(epoch_loss)
    losses.append(avg)
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg:.4f}')

torch.save(model.state_dict(), 'saved/simclr.pt')


## Step 5: Linear Evaluation
Freeze encoder → train a single linear layer with labels. Good representations → high linear accuracy.

In [ ]:
# Load pretrained encoder
model.load_state_dict(torch.load('saved/simclr.pt', map_location=device))
for p in model.encoder.parameters():
    p.requires_grad = False

# Linear probe
linear = nn.Linear(512, 10).to(device)
opt_lin = torch.optim.Adam(linear.parameters(), lr=1e-3)

eval_tf = T.Compose([T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])])
train_set = torchvision.datasets.CIFAR10('./data', train=True,  transform=eval_tf, download=True)
test_set  = torchvision.datasets.CIFAR10('./data', train=False, transform=eval_tf, download=True)
train_ld  = DataLoader(train_set, 256, shuffle=True,  num_workers=2)
test_ld   = DataLoader(test_set,  256, shuffle=False, num_workers=2)

for epoch in range(5):
    linear.train(); model.eval()
    for imgs, labels in train_ld:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            h, _ = model(imgs)
        loss = F.cross_entropy(linear(h), labels)
        opt_lin.zero_grad(); loss.backward(); opt_lin.step()

# Evaluate
correct = total = 0
linear.eval(); model.eval()
with torch.no_grad():
    for imgs, labels in test_ld:
        imgs, labels = imgs.to(device), labels.to(device)
        h, _ = model(imgs)
        preds = linear(h).argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
print(f'Linear Eval Accuracy: {correct/total*100:.2f}%')
